In [ ]:
import pandas as pd
import numpy as np
import joblib
import json
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import MinMaxScaler

DATA_PATH = r"C:\Users\mayil\Downloads\jan to may police violation_anonymized791b166.csv"
MODEL_DIR = "model/"
EARTH_RADIUS_KM = 6371

# -------------------------------
# LOAD + PREPROCESS
# -------------------------------
df = pd.read_csv(DATA_PATH)

df["created_datetime"] = pd.to_datetime(
    df["created_datetime"], utc=True, format="mixed", errors="coerce"
)
df["hour"] = df["created_datetime"].dt.hour

coords = df[["latitude", "longitude"]].to_numpy()
# train.py — replace your DBSCAN config

epsilon = 0.8 / EARTH_RADIUS_KM   # 300 meters — merges nearby streets into one zone
min_samples = 80                   # needs 30+ violations to be a "real" hotspot

dbscan = DBSCAN(
    eps=epsilon,
    min_samples=min_samples,
    algorithm="ball_tree",
    metric="haversine"
)
dbscan.fit(np.radians(coords))

df["cluster"] = dbscan.labels_
df = df[df["cluster"] != -1]

# -------------------------------
# FEATURE ENGINEERING
# -------------------------------
VEHICLE_WEIGHT = {"SCOOTER": 0.3, "CAR": 0.6, "MAXI-CAB": 0.9, "LCV": 0.8}
df["vehicle_weight"] = df["vehicle_type"].map(VEHICLE_WEIGHT).fillna(0.5)

def violation_score(v):
    if "MAIN ROAD" in v or "ROAD CROSSING" in v:
        return 1.0
    if "NO PARKING" in v:
        return 0.7
    return 0.5

df["violation_severity"] = df["violation_type"].astype(str).apply(violation_score)

# -------------------------------
# AGGREGATE HOTSPOTS
# -------------------------------
hotspot_rows = []

for cid, g in df.groupby("cluster"):
    raw_score = (
        0.45 * len(g) +
        0.30 * g["vehicle_weight"].mean() * 100 +
        0.25 * g["violation_severity"].mean() * 100
    )
    peak_hour = int(g["hour"].mode()[0])
    dominant_vehicle = g["vehicle_type"].mode()[0] if "vehicle_type" in g.columns else "UNKNOWN"
    dominant_violation = g["violation_type"].mode()[0] if "violation_type" in g.columns else "UNKNOWN"

    hotspot_rows.append({
        "cluster": cid,
        "latitude": g["latitude"].mean(),
        "longitude": g["longitude"].mean(),
        "raw_score": raw_score,
        "violation_count": len(g),
        "peak_hour": peak_hour,
        "dominant_vehicle": dominant_vehicle,
        "dominant_violation": dominant_violation,
        # Save heatmap points (lat/lng only — very compact)
        # In train.py — replace heatmap_points line inside the for loop
        "heatmap_points": g[["latitude", "longitude", "hour"]].round({"latitude": 5, "longitude": 5}).values.tolist()
    })

hotspots_df = pd.DataFrame(hotspot_rows)

# -------------------------------
# SCALE SCORES
# -------------------------------
scaler = MinMaxScaler(feature_range=(20, 95))
hotspots_df["score"] = scaler.fit_transform(hotspots_df[["raw_score"]])
hotspots_df["score"] = hotspots_df["score"].round(1)

# -------------------------------
# SAVE MODELS
# -------------------------------
joblib.dump(dbscan, MODEL_DIR + "dbscan.pkl")
joblib.dump(scaler, MODEL_DIR + "scaler.pkl")

# -------------------------------
# SAVE PRECOMPUTED HOTSPOTS → tiny file, no CSV needed on server
# -------------------------------
records = hotspots_df.to_dict(orient="records")
with open(MODEL_DIR + "hotspots.json", "w") as f:
    json.dump(records, f)

    

print(f" Training done — {len(records)} hotspots saved to model/hotspots.json")
print(f"   File size: {pd.DataFrame(records).memory_usage(deep=True).sum() / 1024:.1f} KB (approx)")

# After aggregating hotspots, before saving — filter out small clusters
MIN_VIOLATIONS = 250   # a hotspot must have at least 50 violations

records = [r for r in records if r["violation_count"] >= MIN_VIOLATIONS]

with open(MODEL_DIR + "hotspots.json", "w") as f:
    json.dump(records, f)

print(f"{len(records)} meaningful hotspots saved (after filtering)")

 